# KARMAD baseline in EasyTSAD

This notebook implements a paper-faithful EasyTSAD baseline inspired by **KARMAD: KAN-based Adversarial Robust Model for Anomaly Detection**.

Key paper details implemented here:

- KAN-based bidirectional function learning module.
- Shared KAN encoder with two KAN-VAE decoders (`KANsVAE1`, `KANsVAE2`).
- VAE latent distribution with reconstruction loss + KL loss.
- Adversarial training objective following Algorithm 1.
- Paper anomaly score: `0.5 * ||x - O1||^2 + 0.5 * ||x - KANsVAE2(O1)||^2`.
- Window size `5`, latent dimension `8`, depth `4`, learning rate `1e-4`, AdamW optimizer.
- EasyTSAD evaluation: Precision, Recall, F1, AUPRC.

Note: the paper uses SPOT for final thresholding. EasyTSAD evaluation protocols already compute threshold-based metrics and AUPRC from raw anomaly scores, so this notebook returns raw KARMAD anomaly scores to EasyTSAD. A small SPOT-style diagnostic is included but is not required for EasyTSAD evaluation.

In [1]:
# Cell 1 — Setup

!pip install --upgrade pip
!pip install toml torch torchinfo optuna pandas numpy tqdm
!pip install git+https://github.com/CSTCloudOps/EasyTSAD.git

# Clean previous copies
!rm -rf KAN-AD
!rm -rf datasets

# Clone the same dataset structure used in your previous notebooks
!git clone https://github.com/CSTCloudOps/KAN-AD.git
!git clone https://github.com/CSTCloudOps/datasets.git
!mv datasets KAN-AD/datasets

%cd KAN-AD

import os, sys, random, json, math
import numpy as np
import pandas as pd
import torch

project_path = os.getcwd()
if project_path not in sys.path:
    sys.path.append(project_path)

# Fix EasyTSAD import bug if present
!sed -i 's/TSData、/TSData/g; s/TSData,*/TSData/g'     /usr/local/lib/python3.12/dist-packages/EasyTSAD/DataFactory/__init__.py

from EasyTSAD.Controller import TSADController
from EasyTSAD.Methods import BaseMethod

print("✅ Environment ready")
print("Current directory:", project_path)

  Cloning https://github.com/CSTCloudOps/EasyTSAD.git to /tmp/pip-req-build-7v12wlxx
  Running command git clone --filter=blob:none --quiet https://github.com/CSTCloudOps/EasyTSAD.git /tmp/pip-req-build-7v12wlxx
  Resolved https://github.com/CSTCloudOps/EasyTSAD.git to commit 507e5533a142eec7eece4372c55e83bb3faff8a1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into 'KAN-AD'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 18 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 61.77 KiB | 944.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
Cloning into 'datasets'...
remote: Enumerating objects: 4503, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 4503 (delta 2), reused 0 (delta 0), pack-r

In [2]:
# Cell 2 — Reproducibility

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False  # faster on Colab/T4
torch.backends.cudnn.benchmark = True

print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Seed: 42
CUDA available: True
GPU: Tesla T4


In [4]:
# Cell 3 — Dataset inspection

DATA_ROOT = "/content/KAN-AD/datasets/MTS"
DATASETS_TO_RUN = ["SWaT", "MSL", "SMAP"]  # edit this list if needed

for ds in DATASETS_TO_RUN:
    curve_dir = os.path.join(DATA_ROOT, ds, "AllInOne")
    print("Dataset:", ds)
    print("AllInOne exists:", os.path.exists(curve_dir))
    if os.path.exists(curve_dir):
        train = np.load(os.path.join(curve_dir, "train.npy"))
        test = np.load(os.path.join(curve_dir, "test.npy"))
        print("train:", train.shape, "test:", test.shape)

Dataset: SWaT
AllInOne exists: True
train: (473399, 51) test: (449919, 51)
Dataset: MSL
AllInOne exists: True
train: (58317, 55) test: (73729, 55)
Dataset: SMAP
AllInOne exists: True
train: (135183, 25) test: (427617, 25)


In [5]:
# Cell 4 — Window datasets and safety utilities

from torch.utils.data import Dataset, DataLoader


def clean_array(x, clip=1e6):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = np.clip(x, -clip, clip)
    return x.astype(np.float32)


class KARMADWindowDataset(Dataset):
    """
    Paper-style window slicing.
    Window W_t = {x_{t-w+1}, ..., x_t}; each sample is a window.
    The model reconstructs the full window.
    """
    def __init__(self, tsData, phase, window_size):
        self.window_size = int(window_size)
        if phase == "train":
            data = tsData.train
        elif phase == "valid":
            data = tsData.valid
        elif phase == "test":
            data = tsData.test
        else:
            raise ValueError("phase must be train / valid / test")

        self.data = clean_array(data)
        assert self.data.ndim == 2, f"Expected 2D MTS data, got {self.data.shape}"
        assert np.isfinite(self.data).all(), "Data contains non-finite values"

        self.N, self.F = self.data.shape
        self.sample_num = self.N  # include padded early windows, as in paper

    def __len__(self):
        return self.sample_num

    def __getitem__(self, idx):
        w = self.window_size
        if idx < w - 1:
            # Paper: windows shorter than w are padded using x_t
            pad_len = w - 1 - idx
            pad = np.repeat(self.data[idx:idx+1], pad_len, axis=0)
            hist = self.data[:idx+1]
            x = np.concatenate([pad, hist], axis=0)
        else:
            x = self.data[idx-w+1:idx+1]

        return torch.tensor(x, dtype=torch.float32)

print("✅ KARMADWindowDataset ready")

✅ KARMADWindowDataset ready


In [6]:
# Cell 5 — Efficient KAN layers

import torch.nn as nn
import torch.nn.functional as F

class EfficientKANLayer(nn.Module):
    """
    Lightweight KAN-like layer using learnable univariate edge functions.

    The KARMAD paper states that KANs use learnable nonlinear functions on edges,
    often parameterized using spline functions. This implementation uses an RBF
    expansion over each input dimension plus a residual linear path. This keeps the
    KAN idea self-contained and computationally feasible inside Colab/EasyTSAD.
    """
    def __init__(self, in_features, out_features, grid_size=8, grid_range=(-2.0, 2.0)):
        super().__init__()
        self.in_features = int(in_features)
        self.out_features = int(out_features)
        self.grid_size = int(grid_size)

        centers = torch.linspace(grid_range[0], grid_range[1], grid_size)
        self.register_buffer("centers", centers)
        self.log_width = nn.Parameter(torch.zeros(1))

        # basis coefficients: out x in x grid
        self.coeff = nn.Parameter(torch.empty(out_features, in_features, grid_size))
        nn.init.xavier_uniform_(self.coeff)

        self.base = nn.Linear(in_features, out_features)
        self.norm = nn.LayerNorm(out_features)

    def forward(self, x):
        # x: (B, in_features)
        x = torch.nan_to_num(x, nan=0.0, posinf=1e3, neginf=-1e3)
        x = torch.clamp(x, -50.0, 50.0)

        width = F.softplus(self.log_width) + 1e-3
        basis = torch.exp(-((x.unsqueeze(-1) - self.centers) ** 2) / (width ** 2))
        # B x in x grid, coeff out x in x grid -> B x out
        kan_out = torch.einsum("big,oig->bo", basis, self.coeff)
        out = kan_out + self.base(x)
        return self.norm(out)


class KANBlock(nn.Module):
    def __init__(self, dim_in, dim_out, grid_size=8, dropout=0.0):
        super().__init__()
        self.layer = EfficientKANLayer(dim_in, dim_out, grid_size=grid_size)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.act(self.layer(x)))


class KANMLP(nn.Module):
    def __init__(self, dim_in, dim_hidden, dim_out, depth=4, grid_size=8, dropout=0.0):
        super().__init__()
        dims = [dim_in] + [dim_hidden] * max(depth - 1, 1) + [dim_out]
        layers = []
        for i in range(len(dims) - 2):
            layers.append(KANBlock(dims[i], dims[i+1], grid_size=grid_size, dropout=dropout))
        layers.append(EfficientKANLayer(dims[-2], dims[-1], grid_size=grid_size))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

print("✅ Efficient KAN modules ready")

✅ Efficient KAN modules ready


In [7]:
# Cell 6 — KARMAD model modules

class KANsEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=8, depth=4, grid_size=8, dropout=0.0):
        super().__init__()
        self.backbone = KANMLP(input_dim, hidden_dim, hidden_dim, depth=depth, grid_size=grid_size, dropout=dropout)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = self.backbone(x)
        mu = self.mu(h)
        logvar = torch.clamp(self.logvar(h), -8.0, 8.0)
        return mu, logvar


class KANsDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim=64, output_dim=128, depth=4, grid_size=8, dropout=0.0):
        super().__init__()
        self.net = KANMLP(latent_dim, hidden_dim, output_dim, depth=depth, grid_size=grid_size, dropout=dropout)

    def forward(self, z):
        return self.net(z)


def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std


def kl_loss(mu, logvar):
    # KL[N(mu, sigma) || N(0, I)] averaged over batch
    return -0.5 * torch.mean(torch.sum(1.0 + logvar - mu.pow(2) - logvar.exp(), dim=1))


class KANsVAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, deterministic=False):
        mu, logvar = self.encoder(x)
        z = mu if deterministic else reparameterize(mu, logvar)
        out = self.decoder(z)
        return out, mu, logvar

print("✅ KARMAD modules ready")

✅ KARMAD modules ready


In [8]:
# Cell 7 — KARMAD as an EasyTSAD method

import tqdm
from EasyTSAD.Methods import BaseMethod

class KARMAD_TSAD(BaseMethod):
    def __init__(self, params: dict) -> None:
        super().__init__()
        self.__anomaly_score = None

        self.window = int(params.get("window", 5))
        self.batch_size = int(params.get("batch_size", 256))
        self.epochs = int(params.get("epochs", 30))
        self.lr = float(params.get("lr", 1e-4))
        self.patience = int(params.get("patience", 5))
        self.latent_dim = int(params.get("latent_dim", 8))
        self.hidden_dim = int(params.get("hidden_dim", 64))
        self.depth = int(params.get("depth", 4))
        self.grid_size = int(params.get("grid_size", 8))
        self.dropout = float(params.get("dropout", 0.0))
        self.eta = float(params.get("eta", 1.01))  # eta>1 activates the adversarial term progressively
        self.grad_clip = float(params.get("grad_clip", 1.0))

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_ready = False

    def _init_model(self, feature_dim):
        self.feature_dim = int(feature_dim)
        self.input_dim = self.window * self.feature_dim

        self.encoder = KANsEncoder(
            input_dim=self.input_dim,
            hidden_dim=self.hidden_dim,
            latent_dim=self.latent_dim,
            depth=self.depth,
            grid_size=self.grid_size,
            dropout=self.dropout,
        ).to(self.device)

        self.decoder1 = KANsDecoder(
            latent_dim=self.latent_dim,
            hidden_dim=self.hidden_dim,
            output_dim=self.input_dim,
            depth=self.depth,
            grid_size=self.grid_size,
            dropout=self.dropout,
        ).to(self.device)

        self.decoder2 = KANsDecoder(
            latent_dim=self.latent_dim,
            hidden_dim=self.hidden_dim,
            output_dim=self.input_dim,
            depth=self.depth,
            grid_size=self.grid_size,
            dropout=self.dropout,
        ).to(self.device)

        # KANsVAE1 and KANsVAE2 share the same encoder, as described in the paper
        self.vae1 = KANsVAE(self.encoder, self.decoder1).to(self.device)
        self.vae2 = KANsVAE(self.encoder, self.decoder2).to(self.device)

        self.optimizer = torch.optim.AdamW(
            list(self.encoder.parameters()) +
            list(self.decoder1.parameters()) +
            list(self.decoder2.parameters()),
            lr=self.lr,
            weight_decay=1e-5,
        )
        self.model_ready = True

    def _flatten(self, x):
        # x: B x W x F -> B x (W*F)
        x = torch.nan_to_num(x, nan=0.0, posinf=1e3, neginf=-1e3)
        x = torch.clamp(x, -50.0, 50.0)
        return x.reshape(x.shape[0], -1)

    def _losses(self, x_flat, epoch_idx):
        # O1 = KANsVAE1(x), O2 = KANsVAE2(x), Oc2 = KANsVAE2(O1)
        O1, mu_i, logvar_i = self.vae1(x_flat, deterministic=False)
        O2, _, _ = self.vae2(x_flat, deterministic=False)

        # Re-encode O1 through the shared encoder, then decode via decoder2
        mu_o1, logvar_o1 = self.encoder(O1.detach() if False else O1)
        z_o1 = reparameterize(mu_o1, logvar_o1)
        Oc2 = self.decoder2(z_o1)

        rec1 = F.mse_loss(O1, x_flat)
        rec2 = F.mse_loss(O2, x_flat)
        rec3 = F.mse_loss(Oc2, x_flat)

        kl_i = kl_loss(mu_i, logvar_i)
        kl_o1 = kl_loss(mu_o1, logvar_o1)

        L1 = rec1 + kl_i
        L2 = rec2 + kl_i
        L3 = rec3 + kl_o1

        # Paper Algorithm 1: eta^{-n} * L1 + (1-eta^{-n}) * L3, etc.
        # eta > 1 makes eta^{-n} decay gradually from 1.
        weight = self.eta ** (-epoch_idx)
        weight = float(np.clip(weight, 0.0, 1.0))

        L_vae1 = weight * L1 + (1.0 - weight) * L3
        L_vae2 = weight * L2 - (1.0 - weight) * L3

        total = L_vae1 + L_vae2

        return total, {
            "total": total.detach(),
            "L1": L1.detach(),
            "L2": L2.detach(),
            "L3": L3.detach(),
            "rec1": rec1.detach(),
            "rec2": rec2.detach(),
            "rec3": rec3.detach(),
            "kl_i": kl_i.detach(),
            "kl_o1": kl_o1.detach(),
            "adv_weight": torch.tensor(weight),
        }

    def _score_batch(self, x):
        # Paper Eq. 18: delta(x) = 0.5||x-O1||^2 + 0.5||x - KANsVAE2(O1)||^2
        x_flat = self._flatten(x)
        O1, _, _ = self.vae1(x_flat, deterministic=True)
        mu_o1, logvar_o1 = self.encoder(O1)
        z_o1 = mu_o1
        Oc2 = self.decoder2(z_o1)

        err1 = ((x_flat - O1) ** 2).mean(dim=1)
        err2 = ((x_flat - Oc2) ** 2).mean(dim=1)
        score = 0.5 * err1 + 0.5 * err2
        score = torch.nan_to_num(score, nan=1e6, posinf=1e6, neginf=1e6)
        return score

    def train_valid_phase(self, tsTrain):
        train_loader = DataLoader(
            KARMADWindowDataset(tsTrain, "train", self.window),
            batch_size=self.batch_size,
            shuffle=True,
            drop_last=False,
        )
        valid_loader = DataLoader(
            KARMADWindowDataset(tsTrain, "valid", self.window),
            batch_size=self.batch_size,
            shuffle=False,
            drop_last=False,
        )

        sample_x = next(iter(train_loader))
        self._init_model(sample_x.shape[-1])

        best_valid = float("inf")
        best_state = None
        patience_counter = 0
        self.history = []

        for epoch in range(1, self.epochs + 1):
            self.vae1.train(); self.vae2.train(); self.encoder.train()
            train_losses = []

            for x in tqdm.tqdm(train_loader, desc=f"Train {epoch}"):
                x = x.to(self.device)
                x_flat = self._flatten(x)

                self.optimizer.zero_grad(set_to_none=True)
                total_loss, loss_dict = self._losses(x_flat, epoch)

                if not torch.isfinite(total_loss):
                    continue

                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(self.encoder.parameters()) + list(self.decoder1.parameters()) + list(self.decoder2.parameters()),
                    max_norm=self.grad_clip,
                )
                self.optimizer.step()
                train_losses.append(float(total_loss.item()))

            self.vae1.eval(); self.vae2.eval(); self.encoder.eval()
            valid_losses = []
            with torch.no_grad():
                for x in tqdm.tqdm(valid_loader, desc=f"Valid {epoch}"):
                    x = x.to(self.device)
                    x_flat = self._flatten(x)
                    total_loss, _ = self._losses(x_flat, epoch)
                    if torch.isfinite(total_loss):
                        valid_losses.append(float(total_loss.item()))

            train_loss = float(np.mean(train_losses)) if train_losses else float("nan")
            valid_loss = float(np.mean(valid_losses)) if valid_losses else float("nan")

            print(f"Epoch {epoch} | train={train_loss:.6f} | valid={valid_loss:.6f}")
            self.history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss})

            if np.isfinite(valid_loss) and valid_loss < best_valid:
                best_valid = valid_loss
                patience_counter = 0
                best_state = {
                    "encoder": {k: v.detach().cpu().clone() for k, v in self.encoder.state_dict().items()},
                    "decoder1": {k: v.detach().cpu().clone() for k, v in self.decoder1.state_dict().items()},
                    "decoder2": {k: v.detach().cpu().clone() for k, v in self.decoder2.state_dict().items()},
                }
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print("Early stopping")
                    break

        if best_state is not None:
            self.encoder.load_state_dict(best_state["encoder"])
            self.decoder1.load_state_dict(best_state["decoder1"])
            self.decoder2.load_state_dict(best_state["decoder2"])
        else:
            print("⚠️ No finite validation checkpoint saved; using current weights.")

    def test_phase(self, tsData):
        test_loader = DataLoader(
            KARMADWindowDataset(tsData, "test", self.window),
            batch_size=self.batch_size,
            shuffle=False,
            drop_last=False,
        )

        self.vae1.eval(); self.vae2.eval(); self.encoder.eval()
        scores = []
        with torch.no_grad():
            for x in tqdm.tqdm(test_loader, desc="Test"):
                x = x.to(self.device)
                score = self._score_batch(x)
                scores.append(score.cpu().numpy())

        scores = np.concatenate(scores) if scores else np.zeros(len(tsData.test), dtype=np.float64)
        scores = scores[:len(tsData.test)]
        self.__anomaly_score = scores.astype(np.float64)

    def anomaly_score(self) -> np.ndarray:
        return self.__anomaly_score

print("✅ KARMAD_TSAD EasyTSAD method ready")

✅ KARMAD_TSAD EasyTSAD method ready


In [9]:
# Cell 8 — Create KARMAD config

import os

config_text = """[Data_Params]
# KARMAD paper uses min-max normalization.
preprocess = "min-max"
diff_order = 0

[Model_Params.Default]
# Paper hyperparameters
window = 5
latent_dim = 8
depth = 4
lr = 0.0001

# Practical EasyTSAD/Colab settings
batch_size = 256
epochs = 30
patience = 5
hidden_dim = 64
grid_size = 8
dropout = 0.0
grad_clip = 1.0

# Paper Algorithm 1 uses eta^{-n}; eta > 1 gradually activates adversarial component.
eta = 1.01
"""

CFG_PATH = os.path.join("kanad", "config_karmad.toml")
with open(CFG_PATH, "w") as f:
    f.write(config_text)

print("✅ Config written to:", CFG_PATH)
print(open(CFG_PATH).read())

✅ Config written to: kanad/config_karmad.toml
[Data_Params]
# KARMAD paper uses min-max normalization.
preprocess = "min-max"
diff_order = 0

[Model_Params.Default]
# Paper hyperparameters
window = 5
latent_dim = 8
depth = 4
lr = 0.0001

# Practical EasyTSAD/Colab settings
batch_size = 256
epochs = 30
patience = 5
hidden_dim = 64
grid_size = 8
dropout = 0.0
grad_clip = 1.0

# Paper Algorithm 1 uses eta^{-n}; eta > 1 gradually activates adversarial component.
eta = 1.01



In [10]:
# Cell 9 — Run KARMAD experiments

from EasyTSAD.Controller import TSADController

gctrl = TSADController()

gctrl.set_dataset(
    dataset_type="MTS",
    dirname="datasets",
    datasets=DATASETS_TO_RUN,
)

METHOD_NAME = "KARMAD_TSAD"
TRAINING_SCHEMA = "naive"

print("🚀 Running KARMAD baseline on:", DATASETS_TO_RUN)

gctrl.run_exps(
    method=METHOD_NAME,
    training_schema=TRAINING_SCHEMA,
    cfg_path=CFG_PATH,
)

print("✅ KARMAD training finished")

(2026-06-05 23:26:25,914) [INFO]: 
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚══════╝   ╚═╝          ╚═╝   ╚══════╝╚═╝  ╚═╝╚═════╝ 
                                                                      
                         
INFO:logger:
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚═════

🚀 Running KARMAD baseline on: ['SWaT', 'MSL', 'SMAP']


(2026-06-05 23:26:27,369) [INFO]:     [KARMAD_TSAD] handling dataset SWaT | curve AllInOne 
INFO:logger:    [KARMAD_TSAD] handling dataset SWaT | curve AllInOne 
Valid 1: 100%|██████████| 1850/1850 [00:25<00:00, 71.44it/s]


Epoch 1 | train=0.922271 | valid=0.572032


Valid 2: 100%|██████████| 1850/1850 [00:25<00:00, 71.48it/s]


...

Epoch 29 | train=0.024174 | valid=0.024176


Valid 30: 100%|██████████| 529/529 [00:07<00:00, 70.14it/s]


Epoch 30 | train=0.023959 | valid=0.023936


Test: 100%|██████████| 1671/1671 [00:16<00:00, 103.98it/s]

✅ KARMAD training finished


In [11]:
# Cell 10 — Evaluate KARMAD with EasyTSAD metrics

from EasyTSAD.Evaluations.Protocols import (
    EventF1PA,
    PointF1PA,
    PointKthF1PA,
    PointAuprcPA,
)

gctrl.set_evals([
    PointF1PA(),
    EventF1PA(mode="squeeze"),
    PointKthF1PA(k=5),
    PointAuprcPA(),
])

gctrl.do_evals(
    method=METHOD_NAME,
    training_schema=TRAINING_SCHEMA,
)

print("✅ Evaluation completed")

(2026-06-06 00:38:52,899) [INFO]: Register evaluations
INFO:logger:Register evaluations
(2026-06-06 00:38:52,901) [INFO]: Perform evaluations. Method[KARMAD_TSAD], Schema[naive].
INFO:logger:Perform evaluations. Method[KARMAD_TSAD], Schema[naive].
(2026-06-06 00:38:52,903) [INFO]:     [Load Data (All)] DataSets: SWaT,MSL,SMAP 
INFO:logger:    [Load Data (All)] DataSets: SWaT,MSL,SMAP 
(2026-06-06 00:38:53,043) [INFO]:     [KARMAD_TSAD] Eval dataset SWaT <<<
INFO:logger:    [KARMAD_TSAD] Eval dataset SWaT <<<
(2026-06-06 00:38:53,046) [INFO]:         [SWaT] Using default margins (0, 5)
INFO:logger:        [SWaT] Using default margins (0, 5)
(2026-06-06 00:39:03,979) [INFO]:     [KARMAD_TSAD] Eval dataset MSL <<<
INFO:logger:    [KARMAD_TSAD] Eval dataset MSL <<<
(2026-06-06 00:39:03,982) [INFO]:         [MSL] Using default margins (0, 5)
INFO:logger:        [MSL] Using default margins (0, 5)
(2026-06-06 00:39:06,296) [INFO]:     [KARMAD_TSAD] Eval dataset SMAP <<<
INFO:logger:    [KARMA

✅ Evaluation completed


In [13]:
# Cell 11 — Load and summarize KARMAD results

import json, os
import pandas as pd
import numpy as np

rows = []

for ds in DATASETS_TO_RUN:
    avg_path = f"/content/KAN-AD/Results/Evals/{METHOD_NAME}/{TRAINING_SCHEMA}/{ds}/avg.json"

    print("\n" + "=" * 70)
    print(f"Dataset: {ds}")
    print("=" * 70)

    if not os.path.exists(avg_path):
        print("❌ Missing result file:")
        print(avg_path)
        continue

    with open(avg_path, "r") as f:
        avg = json.load(f)

    best_f1 = avg.get("best f1 under pa", {})
    event_f1 = avg.get("event-based f1 under pa with mode squeeze", {})
    delay_f1 = avg.get("best f1 under 5-delay pa", {})

    row = {
        "Model": "KARMAD",
        "Dataset": ds,
        "Precision": best_f1.get("precision", np.nan),
        "Recall": best_f1.get("recall", np.nan),
        "F1": best_f1.get("f1", np.nan),
        "AUPRC": avg.get("point-based auprc pa", np.nan),
        "Event_F1": event_f1.get("f1", np.nan),
        "Delay_F1": delay_f1.get("f1", np.nan),
    }

    rows.append(row)

    # -----------------------------
    # Text format output
    # -----------------------------
    def fmt(x):
        return "-" if pd.isna(x) else f"{x * 100:.2f}%"

    print("Model: KARMAD")
    print(f"Dataset: {ds}")
    print("")
    print("=== AVERAGE RESULTS ===")
    print(f"Precision : {fmt(row['Precision'])}")
    print(f"Recall    : {fmt(row['Recall'])}")
    print(f"F1-score  : {fmt(row['F1'])}")
    print(f"AUPRC     : {fmt(row['AUPRC'])}")
    print(f"Event-F1  : {fmt(row['Event_F1'])}")
    print(f"Delay-F1  : {fmt(row['Delay_F1'])}")

    # Optional thresholds if available
    if "threshold" in best_f1:
        print(f"Best-F1 threshold: {best_f1['threshold']}")
    if "threshold" in event_f1:
        print(f"Event-F1 threshold: {event_f1['threshold']}")
    if "threshold" in delay_f1:
        print(f"Delay-F1 threshold: {delay_f1['threshold']}")

# -----------------------------
# DataFrame summary
# -----------------------------
results_df = pd.DataFrame(rows)

results_pct = results_df.copy()

for c in ["Precision", "Recall", "F1", "AUPRC", "Event_F1", "Delay_F1"]:
    if c in results_pct.columns:
        results_pct[c] = (results_pct[c] * 100).round(2)

print("\n" + "=" * 70)
print("SUMMARY TABLE — KARMAD")
print("=" * 70)

display(results_pct)


Dataset: SWaT
Model: KARMAD
Dataset: SWaT

=== AVERAGE RESULTS ===
Precision : 90.68%
Recall    : 77.96%
F1-score  : 83.84%
AUPRC     : 86.65%
Event-F1  : 10.81%
Delay-F1  : 43.71%
Best-F1 threshold: 0.1918001938611269
Event-F1 threshold: 0.8719043750315905
Delay-F1 threshold: 0.12360206432640553

Dataset: MSL
Model: KARMAD
Dataset: MSL

=== AVERAGE RESULTS ===
Precision : 85.54%
Recall    : 80.67%
F1-score  : 83.03%
AUPRC     : 83.11%
Event-F1  : 9.88%
Delay-F1  : 42.08%
Best-F1 threshold: 0.015313624870032072
Event-F1 threshold: 0.018530435394495726
Delay-F1 threshold: 0.009986504446715117

Dataset: SMAP
Model: KARMAD
Dataset: SMAP

=== AVERAGE RESULTS ===
Precision : 94.50%
Recall    : 54.74%
F1-score  : 69.32%
AUPRC     : 69.37%
Event-F1  : 23.08%
Delay-F1  : 27.60%
Best-F1 threshold: 0.13051564735360444
Event-F1 threshold: 0.2057861213106662
Delay-F1 threshold: 0.0078024466056376696

SUMMARY TABLE — KARMAD


,Model,Dataset,Precision,Recall,F1,AUPRC,Event_F1,Delay_F1
0,KARMAD,SWaT,90.68,77.96,83.84,86.65,10.81,43.71
1,KARMAD,MSL,85.54,80.67,83.03,83.11,9.88,42.08
2,KARMAD,SMAP,94.50,54.74,69.32,69.37,23.08,27.60


In [14]:
# Cell 12 — Export compact row format for your Stage 2 comparison table

for _, r in results_pct.iterrows():
    print(f"Model: KARMAD")
    print(f"Dataset: {r['Dataset']}")
    print(f"Precision: {r['Precision']}")
    print(f"Recall: {r['Recall']}")
    print(f"F1: {r['F1']}")
    print(f"AUPRC: {r['AUPRC']}")
    print()

Model: KARMAD
Dataset: SWaT
Precision: 90.68
Recall: 77.96
F1: 83.84
AUPRC: 86.65

Model: KARMAD
Dataset: MSL
Precision: 85.54
Recall: 80.67
F1: 83.03
AUPRC: 83.11

Model: KARMAD
Dataset: SMAP
Precision: 94.5
Recall: 54.74
F1: 69.32
AUPRC: 69.37



## Notes for reporting

This notebook is designed as a reproducible EasyTSAD adaptation of KARMAD. The paper itself reports Precision, Recall, and F1, but does not report AUPRC in its main comparison table. Here, EasyTSAD additionally computes AUPRC from the raw KARMAD anomaly scores, which makes it compatible with your DeepSKAN comparison table.

If training is slow on Colab T4, reduce `epochs` from 30 to 10 for debugging, then restore it for final results.